# Classical Baselines — Otsu on VH + Majority Class

Two genuine baselines for the Sen1Floods11 test split, both **without any deep learning**:

1. **Majority-class predictor** — always predict "dry". A floor that exists to show *accuracy is meaningless* on a class-imbalanced task.
2. **Otsu thresholding on VH** — the canonical classical SAR flood-detection method. Compute Otsu's threshold per tile on the VH (dB) backscatter; pixels below the threshold are labeled as flood. Water surfaces give very low cross-pol returns (smooth → specular reflection away from sensor), so VH ≪ threshold is a strong physical prior for flood pixels.

This notebook is the **first tier** of the three-tier report comparison: *Otsu → U-Net → (optional) foundation model*. Every metric we report uses the same valid-pixel masking convention as `test.ipynb` so the numbers are directly comparable.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from skimage.filters import threshold_otsu
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT       = Path('.')
DATA_DIR   = ROOT / 'data' / 'sen1floods11' / 'HandLabeled'
S1_DIR     = DATA_DIR / 'S1Hand'
LABEL_DIR  = DATA_DIR / 'LabelHand'
SPLITS_DIR = DATA_DIR / 'splits'

LABEL_NODATA = -1


def load_split_csv(csv_path: Path) -> list:
    df = pd.read_csv(csv_path, header=None, names=['s1', 'label'])
    return list(zip(df['s1'].tolist(), df['label'].tolist()))


test_pairs = load_split_csv(SPLITS_DIR / 'flood_test_data.csv')
print(f'Test tiles: {len(test_pairs)}')

## Otsu prediction function

Per-tile Otsu on VH (in dB). The threshold is computed over valid (non-NaN) pixels only.  
Pixels with VH **below** the threshold are predicted as flood (water → low backscatter).

In [ ]:
def load_tile(s1_fname: str, lbl_fname: str) -> tuple:
    """Read VV, VH (dB) and label. Return (vv, vh, label, valid_mask)."""
    with rasterio.open(S1_DIR / s1_fname) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
    with rasterio.open(LABEL_DIR / lbl_fname) as src:
        label = src.read(1).astype(np.int16)
    sar_valid  = ~np.isnan(vv) & ~np.isnan(vh)
    valid_mask = sar_valid & (label != LABEL_NODATA)
    return vv, vh, label, valid_mask


def otsu_predict(vh: np.ndarray, valid_mask: np.ndarray) -> tuple:
    """Per-tile Otsu on VH. Returns (pred_uint8, threshold_dB).

    Predicts 1 (flood) where VH < threshold AND pixel is valid; 0 otherwise.
    Falls back to all-zeros when there is no valid signal or VH has zero variance.
    """
    pred = np.zeros_like(vh, dtype=np.uint8)
    vh_valid = vh[valid_mask]
    if len(vh_valid) == 0 or vh_valid.min() == vh_valid.max():
        return pred, float('nan')
    thr = threshold_otsu(vh_valid)
    pred[(vh < thr) & valid_mask] = 1
    return pred, float(thr)


# Quick sanity check on tile 0
vv0, vh0, lbl0, vmask0 = load_tile(*test_pairs[0])
pred0, thr0 = otsu_predict(vh0, vmask0)
print(f'Sample tile : {test_pairs[0][0]}')
print(f'  VH range  : [{np.nanmin(vh0):.2f}, {np.nanmax(vh0):.2f}] dB')
print(f'  Otsu thr  : {thr0:.2f} dB')
print(f'  pred flood: {int(pred0.sum())} px / {int(vmask0.sum())} valid')

## Evaluation — accumulate confusion counts over the test split

Single sweep through 90 tiles. For each tile we run **both** baselines (majority-class and Otsu) and accumulate global TP/FP/FN/TN over valid pixels. Metrics are derived from the global counts (micro-averaged), which matches how `BinaryJaccardIndex` etc. behave when updated pixel-by-pixel — so these numbers are directly comparable to `test.ipynb`.

In [ ]:
def confusion_counts(pred: np.ndarray, label: np.ndarray, valid_mask: np.ndarray) -> tuple:
    """Return (tp, fp, fn, tn) over valid pixels for a single tile."""
    p = pred[valid_mask].astype(np.int64)
    g = label[valid_mask].astype(np.int64)
    tp = int(((p == 1) & (g == 1)).sum())
    fp = int(((p == 1) & (g == 0)).sum())
    fn = int(((p == 0) & (g == 1)).sum())
    tn = int(((p == 0) & (g == 0)).sum())
    return tp, fp, fn, tn


def metrics_from_counts(tp: int, fp: int, fn: int, tn: int) -> dict:
    eps = 1e-8
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)
    iou       = tp / (tp + fp + fn + eps)
    dice      = 2 * tp / (2 * tp + fp + fn + eps)
    accuracy  = (tp + tn) / (tp + fp + fn + tn + eps)
    return {'iou': iou, 'dice': dice, 'f1': f1,
            'precision': precision, 'recall': recall, 'accuracy': accuracy}


# accumulators
majority_counts = [0, 0, 0, 0]   # tp, fp, fn, tn
otsu_counts     = [0, 0, 0, 0]
per_tile_otsu   = []             # records for inspection

for s1_fname, lbl_fname in tqdm(test_pairs, desc='Evaluating'):
    _, vh, label, valid_mask = load_tile(s1_fname, lbl_fname)

    # Majority class: always predict 0 (dry)
    maj_pred = np.zeros_like(vh, dtype=np.uint8)
    tp, fp, fn, tn = confusion_counts(maj_pred, label, valid_mask)
    majority_counts = [a + b for a, b in zip(majority_counts, (tp, fp, fn, tn))]

    # Otsu on VH
    otsu_pred, thr = otsu_predict(vh, valid_mask)
    tp, fp, fn, tn = confusion_counts(otsu_pred, label, valid_mask)
    otsu_counts = [a + b for a, b in zip(otsu_counts, (tp, fp, fn, tn))]

    n_valid    = int(valid_mask.sum())
    flood_pct  = 100.0 * label[valid_mask].sum() / max(n_valid, 1)
    tile_iou   = tp / (tp + fp + fn + 1e-8)
    per_tile_otsu.append({
        'tile':       s1_fname.replace('_S1Hand.tif', ''),
        'threshold':  thr,
        'flood_pct':  flood_pct,
        'tile_iou':   tile_iou,
        'n_valid':    n_valid,
    })

majority_metrics = metrics_from_counts(*majority_counts)
otsu_metrics     = metrics_from_counts(*otsu_counts)
otsu_df          = pd.DataFrame(per_tile_otsu)

print()
print('=' * 60)
print('MAJORITY CLASS (always dry)')
print('=' * 60)
for k, v in majority_metrics.items():
    print(f'  {k:<11}: {v:.4f}')

print()
print('=' * 60)
print('OTSU on VH (per-tile threshold)')
print('=' * 60)
for k, v in otsu_metrics.items():
    print(f'  {k:<11}: {v:.4f}')
print()
print(f'  Median Otsu threshold across tiles: {otsu_df["threshold"].median():.2f} dB')
print(f'  Threshold IQR                     : '
      f'[{otsu_df["threshold"].quantile(0.25):.2f}, '
      f'{otsu_df["threshold"].quantile(0.75):.2f}] dB')

print()
print('=' * 60)
print('REFERENCE — trained U-Net ResNet-34 (from test.ipynb)')
print('=' * 60)
trained = {'iou': 0.6622, 'dice': 0.7419, 'f1': 0.7968,
           'precision': 0.8244, 'recall': 0.7710}
for k, v in trained.items():
    print(f'  {k:<11}: {v:.4f}')

## Visual comparison — Otsu vs trained U-Net side by side

Three columns: VH backscatter (Otsu input) | ground truth | Otsu prediction.  
Re-run the cell to draw a different random set of test tiles.

In [ ]:
def percentile_clip(x: np.ndarray, lo: float = 2.0, hi: float = 98.0) -> np.ndarray:
    valid = x[~np.isnan(x)]
    if len(valid) == 0:
        return np.zeros_like(x)
    vmin = np.nanpercentile(x, lo)
    vmax = np.nanpercentile(x, hi)
    out  = np.clip((x - vmin) / (vmax - vmin + 1e-10), 0, 1)
    out[np.isnan(x)] = 0.0
    return out


_rng       = np.random.default_rng()   # no seed — different tiles each run
_indices   = _rng.choice(len(test_pairs), size=3, replace=False).tolist()
_cmap_mask = mcolors.ListedColormap(['lightgray', 'navy', 'white'])

fig, axes = plt.subplots(3, 3, figsize=(13, 13))
axes[0, 0].set_title('VH (dB)',            fontsize=11)
axes[0, 1].set_title('Ground truth',       fontsize=11)
axes[0, 2].set_title('Otsu prediction',    fontsize=11)

for row, tile_idx in enumerate(_indices):
    s1_fname, lbl_fname = test_pairs[tile_idx]
    _, vh, label, valid_mask = load_tile(s1_fname, lbl_fname)
    pred, thr = otsu_predict(vh, valid_mask)

    tp, fp, fn, _ = confusion_counts(pred, label, valid_mask)
    tile_iou  = tp / (tp + fp + fn + 1e-8)
    flood_pct = 100.0 * label[valid_mask].sum() / max(int(valid_mask.sum()), 1)

    gt_disp   = np.where(valid_mask, label.astype(float), 2.0)
    pred_disp = np.where(valid_mask, pred.astype(float), 2.0)

    axes[row, 0].imshow(percentile_clip(vh), cmap='gray')
    axes[row, 0].set_ylabel(
        f'{s1_fname.replace("_S1Hand.tif","")}\n'
        f'thr={thr:.1f}dB  IoU={tile_iou:.3f}\nflood={flood_pct:.1f}%',
        fontsize=8, rotation=0, labelpad=100, va='center')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(gt_disp,   cmap=_cmap_mask, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(pred_disp, cmap=_cmap_mask, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 2].axis('off')

plt.suptitle(
    f'Otsu baseline — random test tiles {_indices}\n'
    'navy=flood  gray=dry  white=nodata',
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

## Per-tile IoU distribution

Where does Otsu break? Cheap diagnostic: histogram of per-tile IoU, and a scatter of IoU vs. flood fraction (Otsu tends to collapse on tiles with very little or no water, where the bimodal assumption fails).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(otsu_df['tile_iou'], bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(otsu_metrics['iou'], color='red', linestyle='--',
                label=f"global IoU = {otsu_metrics['iou']:.3f}")
axes[0].set_xlabel('Per-tile IoU')
axes[0].set_ylabel('Tile count')
axes[0].set_title('Otsu — IoU distribution across test tiles')
axes[0].legend()

sc = axes[1].scatter(
    otsu_df['flood_pct'], otsu_df['tile_iou'],
    c=otsu_df['threshold'], cmap='viridis', s=35, edgecolor='white',
)
axes[1].set_xlabel('Flood pixels in tile (%)')
axes[1].set_ylabel('Per-tile IoU')
axes[1].set_title('Otsu IoU vs. tile flood fraction')
plt.colorbar(sc, ax=axes[1], label='Otsu threshold (dB)')

plt.tight_layout()
plt.show()

# Quick textual summary of failure regimes
n_tiles    = len(otsu_df)
n_zero_iou = int((otsu_df['tile_iou'] < 0.01).sum())
n_no_flood = int((otsu_df['flood_pct'] < 0.1).sum())
print(f'Tiles with IoU < 0.01           : {n_zero_iou} / {n_tiles}')
print(f'Tiles with <0.1% true flood     : {n_no_flood} / {n_tiles}')
print(f'  → on these, Otsu has nothing to find and FP dominates.')